# Math2LaTeX: Complete Training, Evaluation, and Inference Pipeline

This notebook trains our custom **ResNet-18 + Transformer Decoder** model from scratch without any external math OCR services.

### Stages Covered:
1. **Environment & GPU Setup**
2. **Data Pipeline & Augmentation Visualization**
3. **Stage 4: 100-Sample Overfit Sanity Check**
4. **Stage 5: Scaled Training Run with Mixed Precision (fp16)**
5. **Stage 6: Evaluation Metrics (Exact Match, Normalized Edit Distance, Compile Rate)**
6. **Stage 7: Single Image Inference with Beam Search**
7. **Stage 8 & 9: Full Page Photo to LaTeX Pipeline**

In [ ]:
# 1. Environment & GPU Check
import torch
import torchvision
import numpy as np
import cv2

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Visualize Rendered Equations & Realistic Photo Augmentation
import matplotlib.pyplot as plt
from pathlib import Path
from src.data.augment import augment_image
from src.config import load_config

config = load_config("configs/base.yaml")
img_path = list(Path("data/processed/train").glob("*.png"))[0]
clean_img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(3, 1, figsize=(10, 6))
axes[0].imshow(clean_img, cmap='gray')
axes[0].set_title("1. Clean Vector Stroke Rasterization (Height: 128px)")
axes[0].axis('off')

# Augmented photo style 1 (Black ink with notebook lines and shadows)
aug1 = augment_image(clean_img, is_training=True, config=config)
axes[1].imshow(aug1)
axes[1].set_title("2. Photo Augmentation (Camera Tilt, Shadows, Notebook Paper)")
axes[1].axis('off')

# Augmented photo style 2 (Blue ballpoint ink simulation)
aug2 = augment_image(clean_img, is_training=True, config=config)
axes[2].imshow(aug2)
axes[2].set_title("3. Photo Augmentation (Blue Ballpoint, Sensor Noise & Blur)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Run Stage 4: 100-Sample Overfit Milestone
from src.model.train import run_training

print("Starting 100-sample sanity check...")
run_training(
    config_path="configs/base.yaml",
    is_overfit=True,
    epochs_override=40,
    batch_size_override=16,
)

In [ ]:
# 4. Evaluate Model on Unseen Test Split
from src.model.evaluate import evaluate_model, save_worst_mistakes
from src.data.tokenizer import LatexTokenizer
from src.data.dataset import create_dataloader
from src.model.img2latex import Img2Latex

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = LatexTokenizer("data/processed/vocab.json")

ckpt = torch.load("checkpoints/latest_model.pt", map_location=device)
model = Img2Latex(
    vocab_size=tokenizer.vocab_size,
    model_dim=config["model"]["model_dim"],
    decoder_layers=config["model"]["decoder_layers"],
    heads=config["model"]["heads"],
    feedforward_dim=config["model"]["feedforward_dim"],
    dropout=config["model"]["dropout"],
    max_tokens=config["model"]["max_tokens"],
    pad_id=tokenizer.pad_id,
    start_id=tokenizer.start_id,
    end_id=tokenizer.end_id,
    pretrained_backbone=False,
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

test_loader = create_dataloader(
    manifest_path="data/processed/test_manifest.jsonl",
    tokenizer=tokenizer,
    config=config,
    split="test",
    batch_size=16,
    num_workers=0,
    shuffle=False,
)

metrics, results = evaluate_model(model, test_loader, tokenizer, device)
print("=" * 40)
print(f"Exact Match:         {metrics['exact_match']:.2f}%")
print(f"Mean Edit Distance:  {metrics['mean_edit_distance']:.4f}")
print(f"Compile Rate:        {metrics['compile_rate']:.2f}%")
print("=" * 40)

In [ ]:
# 5. Beam Search Single Image Prediction
from src.model.infer import LatexPredictor

predictor = LatexPredictor("checkpoints/latest_model.pt", "configs/base.yaml")
sample_test = list(Path("data/processed/test").glob("*.png"))[0]
pred_latex, conf = predictor.predict(sample_test, beam_size=4)

print(f"Image: {sample_test.name}")
print(f"Predicted LaTeX: {pred_latex}")
print(f"Confidence:     {conf:.2%}")

In [ ]:
# 6. Full Page Photo Pipeline -> .tex Document
from src.pipeline.page_to_tex import PageToTexPipeline

pipeline = PageToTexPipeline("checkpoints/latest_model.pt")
tex_output, details, crops = pipeline.process_page(sample_test, output_tex_path="notebook_output.tex")
print("Generated LaTeX:")
print(tex_output[:600])